In [1]:
# ===================== CELL 1: SETUP & IMPORTS =====================

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2"

FEATURES = {
    'cpm25': 'cpm25.npy',
    'u10': 'u10.npy',
    'v10': 'v10.npy',
    'temp': 't2.npy',
    'rh': 'q2.npy',
    'rain': 'rain.npy',
    'psfc': 'psfc.npy'
}

MONTHS = ['APRIL_16','JULY_16','OCT_16','DEC_16']

print("✅ Setup done")

✅ Setup done


In [2]:
# ===================== CELL 2: LOAD & NORMALIZE =====================

data = {f: [] for f in FEATURES}

for m in MONTHS:
    for f, fname in FEATURES.items():
        path = os.path.join(DATA_DIR, 'raw', m, fname)
        arr = np.load(path).astype(np.float32)
        data[f].append(arr)

for f in data:
    data[f] = np.concatenate(data[f], axis=0)

# normalize using TRAIN stats
stats = {}
for f in data:
    mean = data[f].mean()
    std = data[f].std()
    stats[f] = (mean, std)
    data[f] = (data[f] - mean) / (std + 1e-8)

print("✅ Data ready")

✅ Data ready


In [3]:
class PollutionDatasetV2(Dataset):
    def __init__(self, data):
        self.full = np.stack([data[f] for f in FEATURES], axis=-1)
        self.window = 26
        self.indices = list(range(len(self.full) - self.window))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        sample = self.full[i:i+self.window]

        x = sample[:10]
        y = sample[10:, :, :, 0]
        last_pm = sample[9, :, :, 0]

        x = torch.tensor(x).permute(0,3,1,2).reshape(-1,140,124)
        y = torch.tensor(y)
        last_pm = torch.tensor(last_pm)

        return x.float(), y.float(), last_pm.float()


dataset = PollutionDatasetV2(data)

split = int(0.8 * len(dataset))

train_ds = torch.utils.data.Subset(dataset, list(range(split)))
val_ds   = torch.utils.data.Subset(dataset, list(range(split, len(dataset))))

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)

print("✅ Dataset ready")
print("Train:", len(train_ds), "Val:", len(val_ds))

✅ Dataset ready
Train: 2324 Val: 582


In [4]:
class UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)


class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = UNetBlock(70, 128)
        self.enc2 = UNetBlock(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.mid = UNetBlock(256, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = UNetBlock(384, 128)

        # 🔥 IMPORTANT FIX (restore full resolution)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)

        self.out = nn.Conv2d(64, 16, 1)

    def forward(self, x):
        e1 = self.enc1(x)              # (B,128,140,124)
        e2 = self.enc2(self.pool(e1))  # (B,256,70,62)

        m = self.mid(self.pool(e2))    # (B,256,35,31)

        d = self.up1(m)                # (B,128,70,62)
        d = torch.cat([d, e2], dim=1)  # (B,384,70,62)
        d = self.dec1(d)               # (B,128,70,62)

        d = self.up2(d)                # (B,64,140,124)

        return self.out(d)             # (B,16,140,124)


model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


# 🔥 LOSS FUNCTION (SMAPE + peak focus)
def loss_fn(pred, true):
    smape = torch.abs(pred - true) / (torch.abs(pred) + torch.abs(true) + 1e-8)

    thresh = torch.quantile(true, 0.85)
    weight = 1 + 3 * (true > thresh).float()

    return (smape * weight).mean()


def evaluate(loader):
    model.eval()
    total = 0

    with torch.no_grad():
        for x, y, last in loader:
            x, y, last = x.to(device), y.to(device), last.to(device)

            pred = model(x)
            pred = pred + last.unsqueeze(1)  # residual

            total += loss_fn(pred, y).item()

    return total / len(loader)


# ===================== TRAIN =====================
for epoch in range(12):
    model.train()
    total = 0

    for x, y, last in train_loader:
        x, y, last = x.to(device), y.to(device), last.to(device)

        pred = model(x)
        pred = pred + last.unsqueeze(1)  # residual learning

        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    val_loss = evaluate(val_loader)

    print(f"Epoch {epoch+1}: Train={total/len(train_loader):.4f}, Val={val_loss:.4f}")

Epoch 1: Train=0.3088, Val=0.2281
Epoch 2: Train=0.2625, Val=0.2201
Epoch 3: Train=0.2410, Val=0.1996
Epoch 4: Train=0.2256, Val=0.1957
Epoch 5: Train=0.2158, Val=0.1963
Epoch 6: Train=0.2092, Val=0.1948
Epoch 7: Train=0.2012, Val=0.1873
Epoch 8: Train=0.1940, Val=0.1880
Epoch 9: Train=0.1896, Val=0.1906
Epoch 10: Train=0.1841, Val=0.1849
Epoch 11: Train=0.1805, Val=0.1877
Epoch 12: Train=0.1738, Val=0.1844


In [5]:
# ===================== CELL 5: FINAL INFERENCE =====================

model.eval()

test_data = []

# 🔥 LOAD + NORMALIZE TEST DATA
for f, fname in FEATURES.items():
    path = os.path.join(DATA_DIR, 'test_in', fname)

    arr = np.load(path).astype(np.float32)

    mean, std = stats[f]
    arr = (arr - mean) / (std + 1e-8)

    test_data.append(arr)

# stack → (T, H, W, C)
test = np.stack(test_data, axis=-1)

# reshape → (T, C*10, H, W)
test_tensor = torch.tensor(test).permute(0,1,4,2,3).reshape(test.shape[0], -1, 140, 124)

preds = []

# 🔥 INFERENCE
with torch.no_grad():
    for i in range(0, len(test_tensor), 8):
        batch = test_tensor[i:i+8].to(device)

        out = model(batch)

        # 🔥 residual (last PM2.5 channel)
        last_pm = batch[:, -7, :, :]   # last timestep PM2.5
        out = out + last_pm.unsqueeze(1)

        preds.append(out.cpu().numpy())

preds = np.concatenate(preds, axis=0)

# 🔥 DENORMALIZE
mean, std = stats['cpm25']
preds = preds * std + mean

# 🔥 REMOVE NEGATIVE VALUES
preds = np.clip(preds, 0, None)

# 🔥 FINAL FORMAT (T, H, W, 16)
preds = preds.transpose(0, 2, 3, 1)

# 🔥 SAVE FILE
np.save("/kaggle/working/preds.npy", preds.astype(np.float32))

print("✅ Submission ready!")
print("Shape:", preds.shape)

✅ Submission ready!
Shape: (218, 140, 124, 16)
